# Model 3: Mask R-CNN (Instance Segmentation)\n\nTwo-stage instance segmentation model using torchvision's Mask R-CNN with ResNet50-FPN v2 backbone.\n\n**Architecture**: ResNet50-FPN v2 → RPN → ROI Align → Box Head + Mask Head\n**Training**: SGD with cosine annealing, mixed precision, gradient accumulation\n**Augmentation**: Albumentations (flip, brightness, blur, rotation)

In [ ]:
import torch
import torchvision
from torchvision.models.detection import maskrcnn_resnet50_fpn_v2, MaskRCNN_ResNet50_FPN_V2_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
import os
import json
import numpy as np
import cv2
import time
from collections import defaultdict
from tqdm import tqdm

# =========================
# VERIFY GPU
# =========================

print(f"PyTorch: {torch.__version__}")
print(f"Torchvision: {torchvision.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")

# =========================
# PATHS
# =========================

BASE_DIR = r"C:\Users\User\Desktop\Ipynb"
DATASET_DIR = os.path.join(BASE_DIR, "archive", "dataset_v2")
OUTPUT_DIR = os.path.join(BASE_DIR, "runs", "mask_rcnn")
os.makedirs(OUTPUT_DIR, exist_ok=True)

NUM_CLASSES = 10  # 9 waste categories + 1 background
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Custom TACO Dataset for Mask R-CNN\n\nLoads COCO-format annotations and converts polygon segmentations to binary masks.

In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset, DataLoader
from pycocotools.coco import COCO
from pycocotools import mask as coco_mask_util


class TACODataset(Dataset):
    """TACO dataset for Mask R-CNN with COCO-format annotations."""
    
    def __init__(self, annotation_file, image_dir, transforms=None, max_size=1024):
        self.image_dir = image_dir
        self.transforms = transforms
        self.max_size = max_size
        
        with open(annotation_file) as f:
            data = json.load(f)
        
        self.images = data["images"]
        self.categories = data["categories"]
        
        # Group annotations by image
        self.ann_by_image = defaultdict(list)
        for ann in data["annotations"]:
            self.ann_by_image[ann["image_id"]].append(ann)
        
        # Filter to images that exist on disk
        self.valid_images = []
        for img in self.images:
            img_path = os.path.join(image_dir, img["file_name"])
            if os.path.exists(img_path):
                self.valid_images.append(img)
        
        print(f"  Dataset: {len(self.valid_images)} images, "
              f"{sum(len(self.ann_by_image[img['id']]) for img in self.valid_images)} annotations")
    
    def __len__(self):
        return len(self.valid_images)
    
    def __getitem__(self, idx):
        img_info = self.valid_images[idx]
        img_path = os.path.join(self.image_dir, img_info["file_name"])
        
        # Load image
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        h, w = image.shape[:2]
        
        # Resize preserving aspect ratio
        scale = min(self.max_size / max(h, w), 1.0)
        if scale < 1.0:
            new_h, new_w = int(h * scale), int(w * scale)
            image = cv2.resize(image, (new_w, new_h))
            h, w = new_h, new_w
        else:
            scale = 1.0
        
        # Get annotations
        anns = self.ann_by_image.get(img_info["id"], [])
        
        boxes = []
        labels = []
        masks = []
        areas = []
        
        for ann in anns:
            # Bounding box (COCO: x, y, w, h)
            bx, by, bw, bh = ann["bbox"]
            bx, by, bw, bh = bx * scale, by * scale, bw * scale, bh * scale
            
            if bw < 2 or bh < 2:
                continue
            
            x1, y1 = bx, by
            x2, y2 = bx + bw, by + bh
            
            # Clamp to image bounds
            x1 = max(0, min(x1, w - 1))
            y1 = max(0, min(y1, h - 1))
            x2 = max(x1 + 1, min(x2, w))
            y2 = max(y1 + 1, min(y2, h))
            
            boxes.append([x1, y1, x2, y2])
            labels.append(ann["category_id"])  # already 1-indexed (background=0)
            areas.append((x2 - x1) * (y2 - y1))
            
            # Create binary mask from polygon
            mask = np.zeros((h, w), dtype=np.uint8)
            if "segmentation" in ann and ann["segmentation"]:
                for seg in ann["segmentation"]:
                    pts = np.array(seg, dtype=np.float32).reshape(-1, 2)
                    pts *= scale
                    pts = pts.astype(np.int32)
                    cv2.fillPoly(mask, [pts], 1)
            masks.append(mask)
        
        # Apply augmentation
        if self.transforms and len(boxes) > 0:
            transformed = self.transforms(
                image=image,
                masks=masks,
                bboxes=boxes,
                labels=labels,
            )
            image = transformed["image"]
            masks = transformed["masks"]
            boxes = transformed["bboxes"]
            labels = transformed["labels"]
        
        # Convert to tensors
        if not isinstance(image, torch.Tensor):
            image = torch.as_tensor(image, dtype=torch.float32).permute(2, 0, 1) / 255.0
        
        if len(boxes) == 0:
            target = {
                "boxes": torch.zeros((0, 4), dtype=torch.float32),
                "labels": torch.zeros((0,), dtype=torch.int64),
                "masks": torch.zeros((0, h, w), dtype=torch.uint8),
                "area": torch.zeros((0,), dtype=torch.float32),
                "iscrowd": torch.zeros((0,), dtype=torch.int64),
                "image_id": torch.tensor(img_info["id"]),
            }
        else:
            target = {
                "boxes": torch.as_tensor(boxes, dtype=torch.float32),
                "labels": torch.as_tensor(labels, dtype=torch.int64),
                "masks": torch.stack([torch.as_tensor(m, dtype=torch.uint8) for m in masks]),
                "area": torch.as_tensor(areas, dtype=torch.float32),
                "iscrowd": torch.zeros((len(boxes),), dtype=torch.int64),
                "image_id": torch.tensor(img_info["id"]),
            }
        
        return image, target


def collate_fn(batch):
    """Custom collate for variable-size targets."""
    return tuple(zip(*batch))


print("TACODataset class defined.")

## Create Datasets & DataLoaders

In [ ]:
# =========================
# AUGMENTATION PIPELINES
# =========================

train_transforms = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.1),
    A.RandomBrightnessContrast(p=0.3),
    A.HueSaturationValue(p=0.3),
    A.GaussianBlur(blur_limit=(3, 5), p=0.1),
    A.RandomRotate90(p=0.25),
], bbox_params=A.BboxParams(
    format="pascal_voc",
    label_fields=["labels"],
    min_area=100,
    min_visibility=0.3,
))

# No augmentation for val/test
val_transforms = None

# =========================
# CREATE DATASETS
# =========================

print("Loading datasets...")

train_dataset = TACODataset(
    annotation_file=os.path.join(DATASET_DIR, "train_annotations.json"),
    image_dir=os.path.join(DATASET_DIR, "images", "train"),
    transforms=train_transforms,
    max_size=1024,
)

val_dataset = TACODataset(
    annotation_file=os.path.join(DATASET_DIR, "val_annotations.json"),
    image_dir=os.path.join(DATASET_DIR, "images", "val"),
    transforms=val_transforms,
    max_size=1024,
)

test_dataset = TACODataset(
    annotation_file=os.path.join(DATASET_DIR, "test_annotations.json"),
    image_dir=os.path.join(DATASET_DIR, "images", "test"),
    transforms=val_transforms,
    max_size=1024,
)

# =========================
# DATALOADERS
# =========================

train_loader = DataLoader(
    train_dataset, batch_size=2, shuffle=True,
    collate_fn=collate_fn, num_workers=2, pin_memory=True,
)

val_loader = DataLoader(
    val_dataset, batch_size=2, shuffle=False,
    collate_fn=collate_fn, num_workers=2, pin_memory=True,
)

test_loader = DataLoader(
    test_dataset, batch_size=1, shuffle=False,
    collate_fn=collate_fn, num_workers=2, pin_memory=True,
)

print(f"\nTrain batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader)}")
print(f"Test batches:  {len(test_loader)}")

## Build Mask R-CNN Model

In [ ]:
def build_mask_rcnn(num_classes):
    """Build Mask R-CNN with pretrained ResNet50-FPN v2 backbone."""
    
    # Load pretrained model
    model = maskrcnn_resnet50_fpn_v2(weights=MaskRCNN_ResNet50_FPN_V2_Weights.DEFAULT)
    
    # Replace box predictor head
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    
    # Replace mask predictor head
    in_features_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels
    hidden_layer = 256
    model.roi_heads.mask_predictor = MaskRCNNPredictor(
        in_features_mask, hidden_layer, num_classes
    )
    
    return model


model = build_mask_rcnn(NUM_CLASSES)
model.to(DEVICE)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters:     {total_params / 1e6:.1f}M")
print(f"Trainable parameters: {trainable_params / 1e6:.1f}M")

## Training Loop\n\nFeatures:\n- Mixed precision (AMP) to reduce VRAM usage\n- Gradient accumulation (effective batch size = 2 * 4 = 8)\n- Cosine annealing LR schedule\n- Best checkpoint saving by validation loss\n- Loss logging per epoch

In [ ]:
# =========================
# TRAINING CONFIGURATION
# =========================

NUM_EPOCHS = 50
ACCUMULATION_STEPS = 4  # effective batch = 2 * 4 = 8
LEARNING_RATE = 0.005
MOMENTUM = 0.9
WEIGHT_DECAY = 0.0005
PATIENCE = 15  # early stopping

# Optimizer
optimizer = torch.optim.SGD(
    model.parameters(),
    lr=LEARNING_RATE,
    momentum=MOMENTUM,
    weight_decay=WEIGHT_DECAY,
)

# Cosine annealing scheduler
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=10, T_mult=2, eta_min=1e-6
)

# Mixed precision scaler
scaler = torch.amp.GradScaler("cuda")

print(f"Training config:")
print(f"  Epochs: {NUM_EPOCHS}")
print(f"  Batch size: 2 (x{ACCUMULATION_STEPS} accumulation = effective {2 * ACCUMULATION_STEPS})")
print(f"  LR: {LEARNING_RATE}")
print(f"  Optimizer: SGD + CosineAnnealingWarmRestarts")
print(f"  Mixed precision: enabled")

In [ ]:
# =========================
# TRAINING LOOP
# =========================

import matplotlib.pyplot as plt

train_losses = []
val_losses = []
best_val_loss = float("inf")
patience_counter = 0

for epoch in range(NUM_EPOCHS):
    # --- TRAINING ---
    model.train()
    epoch_loss = 0.0
    num_batches = 0
    optimizer.zero_grad()
    
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [Train]")
    for batch_idx, (images, targets) in enumerate(pbar):
        images = [img.to(DEVICE) for img in images]
        targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]
        
        with torch.amp.autocast("cuda"):
            loss_dict = model(images, targets)
            losses = sum(loss for loss in loss_dict.values())
            losses = losses / ACCUMULATION_STEPS
        
        scaler.scale(losses).backward()
        
        if (batch_idx + 1) % ACCUMULATION_STEPS == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
        
        epoch_loss += losses.item() * ACCUMULATION_STEPS
        num_batches += 1
        pbar.set_postfix(loss=f"{epoch_loss/num_batches:.4f}")
    
    avg_train_loss = epoch_loss / num_batches
    train_losses.append(avg_train_loss)
    
    # --- VALIDATION ---
    model.train()  # Mask R-CNN needs train mode to compute losses
    val_loss = 0.0
    val_batches = 0
    
    with torch.no_grad():
        for images, targets in tqdm(val_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [Val]"):
            images = [img.to(DEVICE) for img in images]
            targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]
            
            with torch.amp.autocast("cuda"):
                loss_dict = model(images, targets)
                losses = sum(loss for loss in loss_dict.values())
            
            val_loss += losses.item()
            val_batches += 1
    
    avg_val_loss = val_loss / val_batches
    val_losses.append(avg_val_loss)
    
    # Update scheduler
    scheduler.step(epoch)
    current_lr = optimizer.param_groups[0]["lr"]
    
    print(f"  Epoch {epoch+1}: train_loss={avg_train_loss:.4f}, "
          f"val_loss={avg_val_loss:.4f}, lr={current_lr:.6f}")
    
    # Save best model
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        torch.save({
            "epoch": epoch + 1,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "val_loss": best_val_loss,
        }, os.path.join(OUTPUT_DIR, "best_mask_rcnn.pth"))
        print(f"  -> Saved best model (val_loss={best_val_loss:.4f})")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"  Early stopping at epoch {epoch+1} (no improvement for {PATIENCE} epochs)")
            break
    
    # Save periodic checkpoint
    if (epoch + 1) % 10 == 0:
        torch.save({
            "epoch": epoch + 1,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "val_loss": avg_val_loss,
        }, os.path.join(OUTPUT_DIR, f"checkpoint_epoch{epoch+1}.pth"))

print(f"\nTraining complete! Best val_loss: {best_val_loss:.4f}")

# =========================
# PLOT TRAINING CURVES
# =========================

fig, ax = plt.subplots(1, 1, figsize=(10, 5))
ax.plot(range(1, len(train_losses) + 1), train_losses, label="Train Loss")
ax.plot(range(1, len(val_losses) + 1), val_losses, label="Val Loss")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title("Mask R-CNN Training Curves")
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "training_curves.png"), dpi=150)
plt.show()

## Evaluate on Test Set\n\nCompute mAP using COCO evaluation, counting accuracy, and inference speed.

In [ ]:
# =========================
# LOAD BEST MODEL
# =========================

checkpoint = torch.load(os.path.join(OUTPUT_DIR, "best_mask_rcnn.pth"), map_location=DEVICE)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()
print(f"Loaded best model from epoch {checkpoint['epoch']} (val_loss={checkpoint['val_loss']:.4f})")

In [ ]:
# =========================
# COCO-STYLE EVALUATION
# =========================

from pycocotools.cocoeval import COCOeval

# Load test annotations as COCO object
with open(os.path.join(DATASET_DIR, "test_annotations.json")) as f:
    test_coco_data = json.load(f)

# Build ground truth counts
gt_counts = defaultdict(int)
for ann in test_coco_data["annotations"]:
    gt_counts[ann["image_id"]] += 1

img_id_by_name = {img["file_name"]: img["id"] for img in test_coco_data["images"]}

# Run inference on test set
coco_results_bbox = []
coco_results_segm = []
count_errors = []
inference_times = []
per_image_results = []

CONF_THRESHOLD = 0.5

test_image_dir = os.path.join(DATASET_DIR, "images", "test")

model.eval()
with torch.no_grad():
    for images, targets in tqdm(test_loader, desc="Test inference"):
        img = images[0].to(DEVICE)
        target = targets[0]
        image_id = target["image_id"].item()
        
        start = time.time()
        with torch.amp.autocast("cuda"):
            predictions = model([img])
        elapsed = (time.time() - start) * 1000
        inference_times.append(elapsed)
        
        pred = predictions[0]
        
        # Filter by confidence
        keep = pred["scores"] >= CONF_THRESHOLD
        boxes = pred["boxes"][keep].cpu().numpy()
        labels = pred["labels"][keep].cpu().numpy()
        scores = pred["scores"][keep].cpu().numpy()
        masks = pred["masks"][keep].cpu().numpy()
        
        gt_count = gt_counts.get(image_id, 0)
        pred_count = len(boxes)
        error = abs(pred_count - gt_count)
        count_errors.append(error)
        
        # Find filename for this image
        fname = ""
        for img_info in test_coco_data["images"]:
            if img_info["id"] == image_id:
                fname = img_info["file_name"]
                break
        
        per_image_results.append({
            "file": fname,
            "gt_count": gt_count,
            "pred_count": pred_count,
            "count_error": error,
            "inference_ms": elapsed,
        })
        
        # Format for COCO evaluation
        h, w = img.shape[1], img.shape[2]
        for i in range(len(boxes)):
            x1, y1, x2, y2 = boxes[i]
            coco_results_bbox.append({
                "image_id": image_id,
                "category_id": int(labels[i]),
                "bbox": [float(x1), float(y1), float(x2 - x1), float(y2 - y1)],
                "score": float(scores[i]),
            })
            
            # Binary mask to RLE
            binary_mask = (masks[i, 0] > 0.5).astype(np.uint8)
            rle = coco_mask_util.encode(np.asfortranarray(binary_mask))
            rle["counts"] = rle["counts"].decode("utf-8")
            coco_results_segm.append({
                "image_id": image_id,
                "category_id": int(labels[i]),
                "segmentation": rle,
                "score": float(scores[i]),
            })

# Run COCO evaluation
from pycocotools.coco import COCO
import tempfile

# Save GT as COCO format
gt_coco = COCO()
gt_coco.dataset = test_coco_data
gt_coco.createIndex()

# Box evaluation
if coco_results_bbox:
    dt_coco_bbox = gt_coco.loadRes(coco_results_bbox)
    coco_eval_bbox = COCOeval(gt_coco, dt_coco_bbox, "bbox")
    coco_eval_bbox.evaluate()
    coco_eval_bbox.accumulate()
    coco_eval_bbox.summarize()
    box_map50 = coco_eval_bbox.stats[1]  # AP@0.50
    box_map50_95 = coco_eval_bbox.stats[0]  # AP@0.50:0.95
else:
    box_map50 = 0.0
    box_map50_95 = 0.0

# Mask evaluation
if coco_results_segm:
    dt_coco_segm = gt_coco.loadRes(coco_results_segm)
    coco_eval_segm = COCOeval(gt_coco, dt_coco_segm, "segm")
    coco_eval_segm.evaluate()
    coco_eval_segm.accumulate()
    coco_eval_segm.summarize()
    mask_map50 = coco_eval_segm.stats[1]
    mask_map50_95 = coco_eval_segm.stats[0]
else:
    mask_map50 = 0.0
    mask_map50_95 = 0.0

within_1 = sum(1 for e in count_errors if e <= 1) / len(count_errors) * 100
within_3 = sum(1 for e in count_errors if e <= 3) / len(count_errors) * 100

print("\n" + "=" * 50)
print("MASK R-CNN - TEST RESULTS")
print("=" * 50)
print(f"Box mAP@50:            {box_map50:.4f}")
print(f"Box mAP@50-95:         {box_map50_95:.4f}")
print(f"Mask mAP@50:           {mask_map50:.4f}")
print(f"Mask mAP@50-95:        {mask_map50_95:.4f}")
print(f"Counting MAE:          {np.mean(count_errors):.2f}")
print(f"Counting within +/-1:  {within_1:.1f}%")
print(f"Counting within +/-3:  {within_3:.1f}%")
print(f"Avg inference time:    {np.mean(inference_times):.1f} ms")
print(f"Median inference time: {np.median(inference_times):.1f} ms")
print("=" * 50)

# Save results
results_data = {
    "metrics": {
        "box_map50": float(box_map50),
        "box_map50_95": float(box_map50_95),
        "mask_map50": float(mask_map50),
        "mask_map50_95": float(mask_map50_95),
        "counting_mae": float(np.mean(count_errors)),
        "counting_within_1": float(within_1),
        "counting_within_3": float(within_3),
        "avg_inference_ms": float(np.mean(inference_times)),
    },
    "per_image": per_image_results,
}

with open(os.path.join(OUTPUT_DIR, "mask_rcnn_results.json"), "w") as f:
    json.dump(results_data, f, indent=2)
print(f"\nResults saved to: {OUTPUT_DIR}/mask_rcnn_results.json")

## Visualize Predictions

In [ ]:
# =========================
# VISUALIZE PREDICTIONS
# =========================

# Category names (1-indexed, 0 = background)
cat_names = {0: "background"}
with open(os.path.join(DATASET_DIR, "test_annotations.json")) as f:
    cat_data = json.load(f)
for cat in cat_data["categories"]:
    cat_names[cat["id"]] = cat["name"]

# Pick 6 sample test images
test_image_files = sorted(os.listdir(os.path.join(DATASET_DIR, "images", "test")))
sample_files = test_image_files[:6]

fig, axes = plt.subplots(2, 3, figsize=(20, 14))
axes = axes.flatten()

np.random.seed(42)
colors = {i: np.random.randint(50, 255, 3).tolist() for i in range(NUM_CLASSES)}

for idx, fname in enumerate(sample_files):
    img_path = os.path.join(DATASET_DIR, "images", "test", fname)
    image = cv2.imread(img_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    h, w = image.shape[:2]
    
    # Resize for model
    scale = min(1024 / max(h, w), 1.0)
    if scale < 1.0:
        image = cv2.resize(image, (int(w * scale), int(h * scale)))
    
    img_tensor = torch.as_tensor(image, dtype=torch.float32).permute(2, 0, 1) / 255.0
    
    with torch.no_grad():
        preds = model([img_tensor.to(DEVICE)])[0]
    
    keep = preds["scores"] >= CONF_THRESHOLD
    boxes = preds["boxes"][keep].cpu().numpy()
    labels = preds["labels"][keep].cpu().numpy()
    scores = preds["scores"][keep].cpu().numpy()
    masks = preds["masks"][keep].cpu().numpy()
    
    # Draw masks and boxes
    overlay = image.copy()
    for i in range(len(boxes)):
        mask = (masks[i, 0] > 0.5).astype(np.uint8)
        color = colors.get(labels[i], [0, 255, 0])
        colored_mask = np.zeros_like(image)
        colored_mask[mask == 1] = color
        overlay = cv2.addWeighted(overlay, 1.0, colored_mask, 0.4, 0)
        
        x1, y1, x2, y2 = boxes[i].astype(int)
        cv2.rectangle(overlay, (x1, y1), (x2, y2), color, 2)
        label_text = f"{cat_names.get(labels[i], '?')} {scores[i]:.2f}"
        cv2.putText(overlay, label_text, (x1, y1 - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
    
    img_id = img_id_by_name.get(fname, -1)
    gt_count = gt_counts.get(img_id, 0)
    
    axes[idx].imshow(overlay)
    axes[idx].set_title(f"GT: {gt_count} | Pred: {len(boxes)}", fontsize=12)
    axes[idx].axis("off")

plt.suptitle("Mask R-CNN Predictions on Test Set", fontsize=16)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "mask_rcnn_predictions.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Visualization saved.")